In [ ]:
import numpy as np
import torch
import gradio as gr
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Load a lightweight, free, local instruction-tuned model (runs smoothly on CPU or GPU)
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

print(f"Loading {MODEL_ID} on {device.upper()}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=dtype).to(device)
model.eval()
print("Local LLM loaded and ready.")


def generate_local_llm(
    system_prompt: str,
    user_prompt: str,
    max_new_tokens: int = 350,
    deterministic: bool = False
) -> str:
    """Run inference locally using Hugging Face chat template."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]
    prompt_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt_text, return_tensors="pt").to(device)

    gen_kwargs = {
        "max_new_tokens": max_new_tokens,
        "pad_token_id": tokenizer.eos_token_id,
    }
    if deterministic:
        # Greedy decoding minimizes hallucinations for strict RAG tasks
        gen_kwargs["do_sample"] = False
    else:
        gen_kwargs.update({
            "do_sample": True,
            "temperature": 0.3,
            "top_p": 0.9
        })

    with torch.no_grad():
        outputs = model.generate(**inputs, **gen_kwargs)

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

Loading Qwen/Qwen2.5-0.5B-Instruct on CPU...


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (2 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Local LLM loaded and ready.


In [ ]:
CROSS_SYSTEM_PROMPT = """
[C - CONTEXT]
You are operating inside the CPGRAMS / JK-IGRAMS Grievance Redressal Co-Pilot for District Administration Kulgam, Jammu & Kashmir. Citizens frequently submit raw, angry, or emotional complaints regarding essential public services.

[R - ROLE]
Act as a senior, empathetic, and impartial e-Governance Grievance Redressal Officer assisting the Deputy Commissioner's Office, District Kulgam.

[O - OBJECTIVE]
Read the raw citizen complaint and produce exactly three clearly labeled sections:
### 1. Neutral Summary
Summarize the factual issue in exactly 2 neutral, objective sentences (remove all anger, insults, and emotional language).

### 2. Sentiment & Urgency Score
- **Citizen Sentiment:** (e.g., Angry, Frustrated, Anxious)
- **Urgency Level:** (Strictly one of: High, Medium, or Low)
- **Reason:** One concise sentence justifying the urgency level.

### 3. Draft Acknowledgment Letter
Write a polite, empathetic, formal acknowledgment letter addressed to the citizen confirming that their grievance has been registered and forwarded to the concerned department in District Kulgam for time-bound verification.

[S - STYLE]
Calm, respectful, citizen-centric, and formal government communication tone.

[S - SPECIFICS]
- NEVER argue with the citizen, mirror their anger, or blame any government official before inquiry.
- NEVER invent fake dates, complaint numbers, or financial promises. Use placeholders like [Grievance_ID] or [Concerned Department] where needed.
- Keep the Neutral Summary strictly to 2 sentences and the letter under 120 words.
""".strip()


def analyze_grievance(raw_complaint: str) -> str:
    """Task 3.1 handler for Gradio: de-escalates and structures a raw complaint."""
    if not raw_complaint or not raw_complaint.strip():
        return "Please enter a citizen complaint to analyze."

    user_prompt = f'Raw Citizen Complaint:\n"""\n{raw_complaint.strip()}\n"""'
    return generate_local_llm(
        system_prompt=CROSS_SYSTEM_PROMPT,
        user_prompt=user_prompt,
        max_new_tokens=380,
        deterministic=False
    )

print("Task 3.1 (C.R.O.S.S. Triage) function initialized.")

Task 3.1 (C.R.O.S.S. Triage) function initialized.


In [ ]:
# Local Government SOP / Scheme Guidelines Repository (District Kulgam)
SOP_KNOWLEDGE_BASE = [
    {
        "sop_id": "SOP-KPDCL-2025-01",
        "title": "KPDCL Distribution Transformer Repair & Replacement SOP (District Kulgam)",
        "content": (
            "1. Reporting: Damage to a distribution transformer (DT) must be reported to the concerned "
            "Assistant Executive Engineer (AEE) Electric Sub-Division or via JK-IGRAMS / CPGRAMS.\n"
            "2. Replacement Timelines (SLA): Under J&K Electricity Supply Code norms, damaged transformers "
            "must be replaced from buffer stock within 24 hours in urban/municipal areas (Kulgam, Yaripora, "
            "Frisal, Qaimoh towns) and within 48 hours in rural/far-flung areas (including D.H. Pora and Kund).\n"
            "3. Zero-Cost Rule: Transportation, crane handling, and repair of government distribution "
            "transformers are borne entirely by KPDCL. Citizens must NOT pay any private transport or repair fee.\n"
            "4. Escalation: If restoration exceeds 48 hours in rural areas, the matter escalates directly to "
            "Executive Engineer (XEN) KPDCL Division Kulgam and Deputy Commissioner's Control Room."
        )
    },
    {
        "sop_id": "SOP-RDD-PMAYG-2025-02",
        "title": "PMAY-G Beneficiary Verification & Exclusion Grievance SOP",
        "content": (
            "1. Eligibility: PMAY-G assistance is strictly restricted to houseless families or families living in "
            "zero-, one-, or two-room kutcha houses listed in the SECC-2011 / Awaas+ Permanent Waitlist (PWL).\n"
            "2. Installment Release: Assistance of Rs. 1.30 lakh (for UT of J&K hilly areas) is released in "
            "3 DBT installments linked to mandatory geo-tagged photographs on the AwaasSoft app (Sanction, Plinth, Completion).\n"
            "3. Wrongful Allotment Inquiry: If a citizen complains that a wealthy/ineligible household received "
            "a PMAY-G shed, the Block Development Officer (BDO) must conduct a spot inquiry within 10 working days "
            "and submit a report to ACD Kulgam for recovery if fraud is confirmed."
        )
    },
    {
        "sop_id": "SOP-IFC-KOUL-2025-03",
        "title": "Irrigation Canal Anti-Encroachment & Desilting SOP",
        "content": (
            "1. Joint Demarcation: Upon receiving an encroachment complaint on an Irrigation Canal, XEN Irrigation & "
            "Flood Control (I&FC) must request joint demarcation from the concerned Tehsildar within 7 days.\n"
            "2. Impartial Removal: Encroachment removal drives must cover the entire demarcated reach of the Canal "
            "uniformly without selective exemption; officers found conducting selective removal face disciplinary inquiry.\n"
            "3. Pre-Sowing Desilting: Routine desilting of irrigation channels must be completed before April 15 each year."
        )
    },
    {
        "sop_id": "SOP-ADM-GFR-2025-04",
        "title": "GFR 2017 & GeM Procurement Violation Inquiry SOP",
        "content": (
            "1. Mandatory Tendering: Hiring of private vehicles or purchase of goods above Rs. 50,000 must follow "
            "GeM portal norms or competitive e-tendering under General Financial Rules (GFR) 2017.\n"
            "2. Inquiry Authority: Any complaint alleging bypassing of GFR norms or arbitrary rejection of genuine bids "
            "is forwarded to ADC / ACR Kulgam or District Vigilance Officer, who must freeze pending payments and "
            "submit a preliminary fact-finding report within 15 days."
        )
    }
]

# Build lightweight TF-IDF Retriever over SOP documents
sop_texts = [f"{doc['title']} {doc['content']}" for doc in SOP_KNOWLEDGE_BASE]
rag_vectorizer = TfidfVectorizer(stop_words="english")
sop_matrix = rag_vectorizer.fit_transform(sop_texts)

RAG_SYSTEM_PROMPT = """
You are a Policy-Compliant Government Assistant for District Kulgam.
STRICT GROUNDING RULES:
1. Answer the citizen's query USING ONLY the facts explicitly written inside the [RETRIEVED SOP] block.
2. Mention the exact SOP ID (e.g., SOP-KPDCL-2025-01) in your answer.
3. Do NOT invent, assume, or add any external rules, compensation amounts, or timelines not stated in [RETRIEVED SOP].
4. If the [RETRIEVED SOP] does NOT contain the answer to the citizen's question, reply ONLY with:
   "This specific detail is not covered in the provided SOP guidelines. Please consult the concerned District Officer for clarification."
""".strip()


def retrieve_relevant_sop(query: str, threshold: float = 0.05):
    """Find the most relevant SOP chunk via TF-IDF cosine similarity."""
    query_vec = rag_vectorizer.transform([query])
    scores = cosine_similarity(query_vec, sop_matrix).flatten()
    best_idx = int(np.argmax(scores))
    best_score = float(scores[best_idx])
    if best_score < threshold:
        return None, best_score
    return SOP_KNOWLEDGE_BASE[best_idx], best_score


def answer_with_mini_rag(citizen_query: str):
    """Task 3.2 handler for Gradio: returns (retrieved_sop_display, grounded_answer)."""
    if not citizen_query or not citizen_query.strip():
        return "No query provided.", "Please enter a policy or scheme question."

    sop_doc, score = retrieve_relevant_sop(citizen_query)

    if sop_doc is None:
        return (
            f"No matching SOP found (max similarity: {score:.2f}).",
            "This specific detail is not covered in the provided SOP guidelines. Please consult the concerned District Officer for clarification."
        )

    retrieved_display = (
        f"**SOP ID:** {sop_doc['sop_id']}\n"
        f"**Title:** {sop_doc['title']}\n"
        f"**Retrieval Similarity Score:** {score:.2f}\n\n"
        f"**Exact Policy Text Passed to LLM:**\n{sop_doc['content']}"
    )

    augmented_prompt = (
        f"[RETRIEVED SOP]\n"
        f"SOP ID: {sop_doc['sop_id']}\n"
        f"Title: {sop_doc['title']}\n"
        f"Guidelines:\n{sop_doc['content']}\n"
        f"[/RETRIEVED SOP]\n\n"
        f"Citizen Query: {citizen_query.strip()}\n\n"
        f"Provide a concise, strictly grounded answer:"
    )

    grounded_answer = generate_local_llm(
        system_prompt=RAG_SYSTEM_PROMPT,
        user_prompt=augmented_prompt,
        max_new_tokens=220,
        deterministic=True
    )

    return retrieved_display, grounded_answer

print(f"Task 3.2 (Mini-RAG) initialized with {len(SOP_KNOWLEDGE_BASE)} Kulgam SOPs.")

Task 3.2 (Mini-RAG) initialized with 4 Kulgam SOPs.


In [ ]:
with gr.Blocks(title="CPGRAMS AI Officer Co-Pilot — District Kulgam") as demo:
    gr.Markdown(
        "# CPGRAMS GenAI & Mini-RAG Officer Co-Pilot (District Kulgam)\n"
        f"**Local Open-Source Model:** `{MODEL_ID}` running on `{device.upper()}` (Zero external APIs)"
    )

    # ---------------------------------------------------------
    # TAB 1: Task 3.1 (C.R.O.S.S. Prompt Engineering)
    # ---------------------------------------------------------
    with gr.Tab("Task 3.1: Complaint Triage (C.R.O.S.S.)"):
        gr.Markdown(
            "Paste a raw or angry citizen complaint to generate a **2-sentence neutral summary**, "
            "**Sentiment/Urgency score**, and a **polite draft acknowledgment letter**."
        )
        complaint_input = gr.Textbox(
            label="Raw Citizen Complaint",
            placeholder="Paste raw citizen grievance here...",
            lines=5
        )
        triage_btn = gr.Button("Analyze & Draft Acknowledgment", variant="primary")
        triage_output = gr.Markdown(label="Officer Co-Pilot Output")

        gr.Examples(
            examples=[
                [
                    "This is absolute nonsense!! Our 100 KVA transformer in Village Bugam, Kulgam burnt "
                    "down 9 days ago and the useless KPDCL staff haven't even picked it up!! 70 families "
                    "are freezing in pitch darkness while paying smart meter bills on time. If not fixed "
                    "by tomorrow we will block the main road!!"
                ],
                [
                    "We are a poor BPL family in Manzgam D.H. Pora and still have not received a shed "
                    "under PMAY-G while rich people with big concrete houses already got funds! Why is "
                    "the BDO office ignoring poor villagers for 8 months?!"
                ]
            ],
            inputs=complaint_input
        )

        triage_btn.click(
            fn=analyze_grievance,
            inputs=complaint_input,
            outputs=triage_output
        )

    # ---------------------------------------------------------
    # TAB 2: Task 3.2 (Mini-RAG Simulation)
    # ---------------------------------------------------------
    with gr.Tab("Task 3.2: Policy-Grounded SOP Assistant (Mini-RAG)"):
        gr.Markdown(
            "Ask a citizen query to retrieve the exact **District Kulgam SOP** and generate a "
            "**strictly grounded, zero-hallucination** officer response."
        )
        rag_query = gr.Textbox(
            label="Citizen Policy / Scheme Query",
            placeholder="Ask a question about transformer timelines, PMAY-G inquiry, Irrigation Koul, or GFR norms...",
            lines=3
        )
        rag_btn = gr.Button("Retrieve SOP & Generate Grounded Answer", variant="primary")

        with gr.Row():
            retrieved_sop_box = gr.Markdown(label="Retrieved SOP Source (Audit Trail)")
            rag_answer_box = gr.Markdown(label="Strictly Grounded LLM Response")

        gr.Examples(
            examples=[
                # In-SOP Query (Facts present in SOP-KPDCL-2025-01)
                [
                    "Our transformer in rural D.H. Pora is damaged. Within how many hours must KPDCL "
                    "replace it, and do villagers have to pay for crane transport?"
                ],
                # In-SOP Query (Facts present in SOP-RDD-PMAYG-2025-02)
                [
                    "A rich family was wrongly allotted a PMAY-G house in our village. Who conducts the "
                    "inquiry and within how many working days?"
                ],
                # Hallucination Trap Query (NOT covered in SOP)
                [
                    "Because of a 5-day transformer failure in Yaripora, my refrigerator food spoiled "
                    "and my TV got damaged. How much cash compensation will KPDCL pay for my TV and food?"
                ]
            ],
            inputs=rag_query
        )

        rag_btn.click(
            fn=answer_with_mini_rag,
            inputs=rag_query,
            outputs=[retrieved_sop_box, rag_answer_box]
        )

# Launch inline inside Jupyter Notebook and print local browser URL
demo.launch(inline=True, share=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>